# Narkomfin Type F — Three-Floor Spatial Analysis (Grid Sampling)

Analyze L1, L2, and L3 of the Narkomfin Type F apartment as a single connected building.

**Method:** Load single OBJ → separate floor faces by Z-level → grid-sample each floor → build per-floor graphs → union → stitch with stair edges at each floor pair → run spatial analysis across all three floors.

**Stair geometry** is in the same OBJ — surfaces that span multiple Z-levels are identified automatically. All 19 stairs span the full height (Z=0 to Z=6), so each stair gets connections at both the L1→L2 and L2→L3 junctions.

## 1. Imports

In [ ]:
import time
import numpy as np

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Color import Color

print(Helper.Version())

In [ ]:
renderer = "vscode"

## 2. Configuration

In [ ]:
from pathlib import Path

HERE = Path.cwd()
ASSET_DIR  = HERE.parent / '02_graph_analysis' / 'assets'
EXPORT_DIR = HERE.parent / '03_node_classification' / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

OBJ_FILE = ASSET_DIR / 'TheNarkomfinHouse-Ftype-withStairs.obj'

GRID_SIZE     = 0.5
FLOOR_HEIGHTS = [0, 3, 6]
FLOOR_NAMES   = ['L1 (Z=0)', 'L2 (Z=3)', 'L3 (Z=6)']
FLOOR_Z_VIS   = [0, 12, 24]

print(f'OBJ: {OBJ_FILE.name} — exists: {OBJ_FILE.exists()}')
print(f'Grid size: {GRID_SIZE}')
print(f'Floors: {len(FLOOR_HEIGHTS)}')

## 3. Utility functions

In [ ]:
from matplotlib.path import Path as MplPath
import plotly.graph_objects as go

def points_inside_faces(face_list, test_pts):
    """Ray-casting point-in-polygon test (handles concave polygons correctly)."""
    inside = np.zeros(len(test_pts), bool)
    for f in face_list:
        vs = Topology.Vertices(f)
        poly = np.array([(Vertex.X(v), Vertex.Y(v)) for v in vs])
        if len(poly) >= 3:
            path = MplPath(poly)
            inside |= path.contains_points(test_pts)
    return inside

def find_closest_node(node_xy, x, y):
    d = (node_xy[:, 0] - x) ** 2 + (node_xy[:, 1] - y) ** 2
    return int(d.argmin())

def rk(u, v):
    return (round(float(u), 3), round(float(v), 3))

def make_cell_face(cx, cy, cz, h):
    pts = [Vertex.ByCoordinates(cx - h, cy - h, cz),
           Vertex.ByCoordinates(cx + h, cy - h, cz),
           Vertex.ByCoordinates(cx + h, cy + h, cz),
           Vertex.ByCoordinates(cx - h, cy + h, cz)]
    return Face.ByWire(Wire.ByVertices(pts, close=True))

def show_ortho(fig):
    fig.update_layout(
        scene_camera=dict(
            eye=dict(x=1.6, y=-1.6, z=1.2),
            up=dict(x=0, y=0, z=1),
            projection=dict(type="orthographic")
        ),
        scene=dict(aspectmode="data"),
        autosize=True,
        margin=dict(l=10, r=10, t=10, b=10)
    )
    fig.show(renderer=renderer)

print('Utilities loaded (ray-casting PIP).')

## 4. Load OBJ and separate floor faces from stairs

In [ ]:
result = Topology.ByOBJPath(str(OBJ_FILE))
all_loaded = []
if isinstance(result, list):
    for item in result:
        if Topology.IsInstance(item, 'Cluster'):
            cf = Topology.Faces(item)
            if cf: all_loaded.extend(cf)
        elif Topology.IsInstance(item, 'Face'):
            all_loaded.append(item)
else:
    all_loaded = Topology.Faces(result)
print(f'Total faces loaded from OBJ: {len(all_loaded)}')

floor_faces = {h: [] for h in FLOOR_HEIGHTS}
stair_surfaces = []
other_faces = []

for f in all_loaded:
    zs = set(round(Vertex.Z(v), 1) for v in Topology.Vertices(f))
    if len(zs) == 1:
        z = zs.pop()
        matched = False
        for h in FLOOR_HEIGHTS:
            if abs(z - h) < 0.5:
                floor_faces[h].append(f)
                matched = True
                break
        if not matched:
            other_faces.append(f)
    else:
        stair_surfaces.append(f)

print('\nFloor faces by level:')
for h, name in zip(FLOOR_HEIGHTS, FLOOR_NAMES):
    print(f'  {name}: {len(floor_faces[h])} faces')
print(f'\nStair surfaces (multi-Z): {len(stair_surfaces)}')
if other_faces:
    print(f'Other faces (unclassified): {len(other_faces)}')

stair_locations = []
for f in stair_surfaces:
    verts = Topology.Vertices(f)
    xs = [Vertex.X(v) for v in verts]
    ys = [Vertex.Y(v) for v in verts]
    stair_locations.append(((min(xs)+max(xs))/2, (min(ys)+max(ys))/2))
    zs = sorted(set(round(Vertex.Z(v), 1) for v in verts))

print(f'\nStair plan locations:')
for i, (sx, sy) in enumerate(stair_locations[:5]):
    print(f'  Stair {i+1}: X={sx:.2f}, Y={sy:.2f}')
if len(stair_locations) > 5:
    print(f'  ... and {len(stair_locations)-5} more')

## 5. Show raw floor plans

In [ ]:
for h, name in zip(FLOOR_HEIGHTS, FLOOR_NAMES):
    if not floor_faces[h]:
        print(f'{name}: no faces — skipping')
        continue
    cluster = Cluster.ByTopologies(floor_faces[h])
    print(f'{name}: {len(floor_faces[h])} faces')
    fig = Topology.Show(cluster,
                  showFigure=False,
                  faceColor=[210,210,250],
                  faceOpacity=1,
                  edgeColor='white',
                  edgeWidth=2,
                  showVertices=False,
                  backgroundColor='black',
                  width=900, height=400,
                  renderer=renderer)
    show_ortho(fig)

## 6. Grid-sample navigable points on each floor

In [ ]:
all_xs, all_ys = [], []
for h in FLOOR_HEIGHTS:
    for f in floor_faces[h]:
        for v in Topology.Vertices(f):
            all_xs.append(Vertex.X(v))
            all_ys.append(Vertex.Y(v))
UMIN, UMAX = min(all_xs), max(all_xs)
VMIN, VMAX = min(all_ys), max(all_ys)
print(f'Plan bounding box: X[{UMIN:.1f}, {UMAX:.1f}]  Y[{VMIN:.1f}, {VMAX:.1f}]')
print(f'Plan size: {UMAX-UMIN:.1f} x {VMAX-VMIN:.1f}')

us = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
vs = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)
UU, VV = np.meshgrid(us, vs)
GRID_PTS = np.column_stack([UU.ravel(), VV.ravel()])
print(f'Grid: {len(us)}x{len(vs)} = {len(GRID_PTS)} candidate points')

floor_valid = {}
for h, name in zip(FLOOR_HEIGHTS, FLOOR_NAMES):
    mask = points_inside_faces(floor_faces[h], GRID_PTS)
    floor_valid[h] = GRID_PTS[mask]
    print(f'  {name}: {len(floor_valid[h])} navigable nodes')

## 6b. Grid overlay on floor plans
Show the candidate grid points overlaid on each floor — filled (navigable) vs empty (outside).

In [ ]:
for h, name in zip(FLOOR_HEIGHTS, FLOOR_NAMES):
    valid = floor_valid[h]
    if len(valid) == 0:
        print(f'{name}: 0 navigable points — skipping')
        continue
    grid_verts = [Vertex.ByCoordinates(float(u), float(v), 0) for u, v in valid]
    grid_cluster = Cluster.ByTopologies(grid_verts)
    floor_cluster = Cluster.ByTopologies(floor_faces[h])
    print(f'{name}: {len(valid)} navigable points')
    fig = Topology.Show(floor_cluster, grid_cluster,
                  showFigure=False,
                  faceColor=[210,210,250],
                  faceOpacity=0.5,
                  edgeColor='white',
                  edgeWidth=1,
                  vertexSize=2,
                  vertexColor='red',
                  backgroundColor='black',
                  width=900, height=400,
                  renderer=renderer)
    show_ortho(fig)

## 7. Build per-floor graphs + display cells, then stack

In [ ]:
all_v = []
all_e = []
floor_index_map = {}
display_faces = []
cell_lookup = {}
H = GRID_SIZE / 2.0

for fi, h in enumerate(FLOOR_HEIGHTS):
    valid = floor_valid[h]
    z_vis = FLOOR_Z_VIS[fi]
    idx = {}
    for (u, v) in valid:
        key = rk(u, v)
        idx[key] = len(all_v)
        all_v.append(Vertex.ByCoordinates(float(u), float(v), float(z_vis)))
        cell = make_cell_face(float(u), float(v), float(z_vis), H)
        display_faces.append(cell)
        cell_lookup[(fi, key)] = cell
    floor_index_map[h] = idx
    ne = 0
    for (u, v) in valid:
        for du, dv in [(GRID_SIZE, 0), (0, GRID_SIZE)]:
            k = rk(u + du, v + dv)
            if k in idx:
                all_e.append(Edge.ByVertices([all_v[idx[rk(u, v)]], all_v[idx[k]]]))
                ne += 1
    print(f'  {FLOOR_NAMES[fi]}: {len(valid)} nodes, {ne} horizontal edges')
print(f'Subtotal: {len(all_v)} nodes, {len(all_e)} horizontal edges')

## 7b. Discretized floor plans (cell faces)
Each navigable grid point becomes a square cell.

In [ ]:
for fi, (h, name) in enumerate(zip(FLOOR_HEIGHTS, FLOOR_NAMES)):
    cells_this_floor = [cell_lookup[(fi, rk(u, v))]
                        for u, v in floor_valid[h]
                        if (fi, rk(u, v)) in cell_lookup]
    if not cells_this_floor:
        print(f'{name}: 0 cells — skipping')
        continue
    print(f'{name}: {len(cells_this_floor)} cells')
    fig = Topology.Show(cells_this_floor,
                  showFigure=False,
                  faceColor=[210,210,250],
                  faceOpacity=1,
                  edgeColor='grey',
                  edgeWidth=0.5,
                  showVertices=False,
                  backgroundColor='black',
                  width=900, height=400,
                  renderer=renderer)
    show_ortho(fig)

## 8. Connect floors through stair edges

In [ ]:
stair_edges_added = 0

for sx, sy in stair_locations:
    for i in range(len(FLOOR_HEIGHTS) - 1):
        lv_low = FLOOR_HEIGHTS[i]
        lv_high = FLOOR_HEIGHTS[i + 1]
        valid_low = floor_valid[lv_low]
        valid_high = floor_valid[lv_high]
        idx_low = floor_index_map[lv_low]
        idx_high = floor_index_map[lv_high]

        if len(valid_low) == 0 or len(valid_high) == 0:
            continue

        i1 = find_closest_node(valid_low, sx, sy)
        i2 = find_closest_node(valid_high, sx, sy)
        key1 = rk(valid_low[i1, 0], valid_low[i1, 1])
        key2 = rk(valid_high[i2, 0], valid_high[i2, 1])
        gi1 = idx_low[key1]
        gi2 = idx_high[key2]
        all_e.append(Edge.ByVertices([all_v[gi1], all_v[gi2]]))
        stair_edges_added += 1

print(f'Added {stair_edges_added} stair edges from {len(stair_locations)} stair locations')
print(f'  (each stair connects {len(FLOOR_HEIGHTS)-1} floor pairs)')
print(f'Total: {len(all_v)} nodes, {len(all_e)} edges')

## 9. Build the combined building graph

In [ ]:
t0 = time.time()
building_graph = Graph.ByVerticesEdges(all_v, all_e)
gverts = Graph.Vertices(building_graph)
gedges = Graph.Edges(building_graph)
print(f'Building graph: {len(gverts)} vertices, {len(gedges)} edges  ({time.time()-t0:.1f}s)')
print(f'Graph density: {Graph.Density(building_graph):.6f}')

## 10. Show the building graph

In [ ]:
fig = Topology.Show(building_graph,
              showFigure=False,
              vertexSize=2,
              vertexColor='red',
              edgeColor='lightgrey',
              backgroundColor='black',
              width=900, height=600,
              renderer=renderer)
show_ortho(fig)

## 11. Spatial Analysis — heatmap helper

In [ ]:
def heatmap_from_values(values, title, colorScale='thermal'):
    """Pair each graph vertex with its display cell, colour by value, render."""
    face_val_pairs = []
    for v, val in zip(gverts, values):
        z = Vertex.Z(v)
        fi = None
        for idx, zv in enumerate(FLOOR_Z_VIS):
            if abs(round(z) - zv) < 1:
                fi = idx
                break
        if fi is None:
            continue
        key = rk(Vertex.X(v), Vertex.Y(v))
        cell = cell_lookup.get((fi, key))
        if cell is not None:
            face_val_pairs.append((cell, val))

    vals = [v for _, v in face_val_pairs]
    mn, mx = float(min(vals)), float(max(vals))
    if mx == mn: mx = mn + 1e-9
    for f, val in face_val_pairs:
        col = Color.AnyToHex(Color.ByValueInRange(float(val), minValue=mn, maxValue=mx, colorScale=colorScale))
        d = Topology.Dictionary(f)
        d = Dictionary.SetValueAtKey(d, 'hm_color', col)
        Topology.SetDictionary(f, d)
    faces = [f for f, _ in face_val_pairs]
    print(f'{title}: {len(faces)} cells, range [{mn:.4f}, {mx:.4f}]')
    fig = Topology.Show(faces,
                  showFigure=False,
                  faceColorKey='hm_color',
                  faceOpacity=1,
                  showEdges=False,
                  showVertices=False,
                  backgroundColor='black',
                  width=900, height=600,
                  renderer=renderer)
    show_ortho(fig)

print('Heatmap helper ready.')

## 12. Shortest Path — Topological (red) vs Geometric (blue)
Cross-floor path from L1 to L3. Red = graph edges (grid steps). Blue = geometrically straightened within each floor's boundary.

In [ ]:
l1_xy = floor_valid[FLOOR_HEIGHTS[0]]
l3_xy = floor_valid[FLOOR_HEIGHTS[2]]
l1_idx = floor_index_map[FLOOR_HEIGHTS[0]]
l3_idx = floor_index_map[FLOOR_HEIGHTS[2]]

i_start = find_closest_node(l1_xy, UMIN + 2, VMAX - 2)
start_key = rk(l1_xy[i_start, 0], l1_xy[i_start, 1])
start_v = all_v[l1_idx[start_key]]

i_end = find_closest_node(l3_xy, UMAX - 2, VMIN + 2)
end_key = rk(l3_xy[i_end, 0], l3_xy[i_end, 1])
end_v = all_v[l3_idx[end_key]]

print(f'Start (L1): {Vertex.X(start_v):.1f}, {Vertex.Y(start_v):.1f}')
print(f'End   (L3): {Vertex.X(end_v):.1f}, {Vertex.Y(end_v):.1f}')

t0 = time.time()
shortest_path = Graph.ShortestPath(building_graph, vertexA=start_v, vertexB=end_v)
print(f'Shortest path computed in {time.time()-t0:.2f}s')

if shortest_path:
    topo_len = Wire.Length(shortest_path)
    print(f'  Topological path length (red): {topo_len:.2f} units')
    for edge in Topology.Edges(shortest_path):
        edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [5,'red']))

    path_verts = Topology.Vertices(shortest_path)
    straight_wires = []
    straight_total = 0
    for fi, (h, name) in enumerate(zip(FLOOR_HEIGHTS, FLOOR_NAMES)):
        z_vis = FLOOR_Z_VIS[fi]
        seg_verts = [v for v in path_verts if abs(Vertex.Z(v) - z_vis) < 0.1]
        if len(seg_verts) >= 2 and len(floor_faces[h]) > 0:
            seg_wire = Wire.ByVertices(seg_verts, close=False)
            host = floor_faces[h][0]
            straight = Wire.Straighten(seg_wire, host=host)
            if straight:
                slen = Wire.Length(straight)
                straight_total += slen
                print(f'  {name} straightened (blue): {slen:.2f} units')
                for e in Topology.Edges(straight):
                    e = Topology.SetDictionary(e, Dictionary.ByKeysValues(['width','color'], [4,'blue']))
                straight_wires.append(straight)

    stair_dist = sum(abs(FLOOR_Z_VIS[i+1] - FLOOR_Z_VIS[i]) for i in range(len(FLOOR_Z_VIS)-1)
                     if any(abs(Vertex.Z(v) - FLOOR_Z_VIS[i]) < 0.1 for v in path_verts)
                     and any(abs(Vertex.Z(v) - FLOOR_Z_VIS[i+1]) < 0.1 for v in path_verts))
    straight_total += stair_dist
    savings = (1 - straight_total / topo_len) * 100 if topo_len > 0 else 0
    print(f'  Straightened total (blue + stairs): {straight_total:.2f} units')
    print(f'  Savings vs topological: {savings:.1f}%')

    show_items = [building_graph, shortest_path] + straight_wires
    fig = Topology.Show(*show_items,
                  showFigure=False,
                  vertexSize=1,
                  vertexColor='grey',
                  edgeColor='darkgrey',
                  edgeColorKey='color',
                  edgeWidthKey='width',
                  backgroundColor='black',
                  width=900, height=600,
                  renderer=renderer)
    show_ortho(fig)
else:
    print('  No path found — check stair connections.')

## 13. Degree Centrality

In [ ]:
degree_values = Graph.DegreeCentrality(building_graph)
heatmap_from_values(degree_values, 'Degree Centrality')

## 14. Closeness Centrality / Integration
Measures how close a node is to all other nodes — corresponds to **global integration** in space syntax.

In [ ]:
closeness_values = Graph.ClosenessCentrality(building_graph, colorScale='thermal')
heatmap_from_values(closeness_values, 'Closeness Centrality')

## 15. Betweenness Centrality / Choice
Measures how often a node lies on shortest paths between other nodes — identifies circulation bottlenecks.

In [ ]:
betweenness_values = Graph.BetweennessCentrality(building_graph, normalize=True, colorScale='thermal')
heatmap_from_values(betweenness_values, 'Betweenness Centrality')

## 16. Clustering Coefficient
Measures how interconnected a node's neighbors are.

In [ ]:
clustering_values = Graph.LocalClusteringCoefficient(building_graph)
heatmap_from_values(clustering_values, 'Clustering Coefficient')

## 17. Graph Diameter
The longest shortest path between any two nodes. **Very slow** at fine grids — set `RUN_DIAMETER = True` to run.

In [ ]:
RUN_DIAMETER = False

if RUN_DIAMETER:
    t0 = time.time()
    diameter = Graph.Diameter(building_graph)
    print(f'Graph diameter: {diameter} steps  ({time.time()-t0:.1f}s)')
    print(f'Graph density:  {Graph.Density(building_graph):.6f}')
    print(f'Total vertices: {len(gverts)}')
    print(f'Total edges:    {len(gedges)}')
else:
    print('Diameter skipped (set RUN_DIAMETER = True to enable).')
    print('Consider running at GRID_SIZE=1.0 in a separate notebook for this.')

## 18. Community Detection (Louvain Method)
Partitions the graph into communities of tightly connected nodes — reveals functional spatial zones.

In [ ]:
t0 = time.time()
community_list = Graph.CommunityPartition(building_graph, colorScale="viridis")
print(f'Community partition computed in {time.time()-t0:.1f}s')

n_communities = len(set(community_list))
print(f'Number of communities: {n_communities}')

for v in gverts:
    d = Topology.Dictionary(v)
    cp_color = Dictionary.ValueAtKey(d, "cp_color")
    z = Vertex.Z(v)
    fi = None
    for idx, zv in enumerate(FLOOR_Z_VIS):
        if abs(round(z) - zv) < 1:
            fi = idx
            break
    if fi is None:
        continue
    key = rk(Vertex.X(v), Vertex.Y(v))
    cell = cell_lookup.get((fi, key))
    if cell is not None and cp_color is not None:
        cd = Topology.Dictionary(cell)
        cd = Dictionary.SetValueAtKey(cd, 'cp_color', cp_color)
        Topology.SetDictionary(cell, cd)

all_cells = [cell_lookup[(fi, rk(u, v))]
             for fi, h in enumerate(FLOOR_HEIGHTS)
             for u, v in floor_valid[h]
             if (fi, rk(u, v)) in cell_lookup]

fig = Topology.Show(all_cells,
              showFigure=False,
              faceColorKey='cp_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              backgroundColor='black',
              width=900, height=600,
              renderer=renderer)
show_ortho(fig)

## 19. Isovists (per floor)
Compute isovists from a coarse grid of viewpoints on each floor. Set `RUN_ISOVISTS = True` to run (slow).

In [ ]:
RUN_ISOVISTS = False

if RUN_ISOVISTS:
    from topologicpy.Grid import Grid

    for fi, (h, name) in enumerate(zip(FLOOR_HEIGHTS, FLOOR_NAMES)):
        faces = floor_faces[h]
        if len(faces) == 0:
            print(f'{name}: no faces — skipping isovists')
            continue

        z_vis = FLOOR_Z_VIS[fi]
        floor_gverts = [v for v in gverts if abs(round(Vertex.Z(v)) - z_vis) < 1]
        print(f'\n=== {name} isovists ===')
        print(f'  Graph vertices on this floor: {len(floor_gverts)}')

        all_iso_verts = []
        all_vis_values = []

        for room_i, gallery in enumerate(faces):
            b_r = Wire.BoundingRectangle(gallery)
            if b_r is None:
                continue
            bd = Topology.Dictionary(b_r)
            w = Dictionary.ValueAtKey(bd, "width")
            l = Dictionary.ValueAtKey(bd, "length")
            if w is None or l is None or w < 1 or l < 1:
                continue

            iso_grid = Grid.VerticesByDistances(gallery, clip=True,
                                                 uRange=list(range(0, int(w)+10, 10)),
                                                 vRange=list(range(0, int(l)+10, 10)))
            if iso_grid is None:
                continue
            iso_verts = Topology.Vertices(iso_grid)
            if not iso_verts:
                continue

            t0 = time.time()
            for iv in iso_verts:
                isovist = Face.Isovist(gallery, iv)
                if isovist:
                    b_list = Vertex.IsInternal2D(floor_gverts, isovist)
                    n = sum(1 for b in b_list if b)
                    d = Dictionary.ByKeyValue("visibility", n)
                    iv = Topology.SetDictionary(iv, d)
                    all_iso_verts.append(iv)
                    all_vis_values.append(n)

            print(f'  Room {room_i+1}/{len(faces)}: {len(iso_verts)} viewpoints ({time.time()-t0:.0f}s)')

        if all_vis_values:
            mn_v, mx_v = min(all_vis_values), max(all_vis_values)
            print(f'  Visibility range: [{mn_v}, {mx_v}]')

            for v in floor_gverts:
                Vertex.InterpolateValue(v, vertices=all_iso_verts, n=2, key="visibility")

            vis_cells = []
            for v in floor_gverts:
                d = Topology.Dictionary(v)
                vb = Dictionary.ValueAtKey(d, "visibility")
                if vb is None: vb = 0
                key = rk(Vertex.X(v), Vertex.Y(v))
                cell = cell_lookup.get((fi, key))
                if cell is not None:
                    col = Color.AnyToHex(Color.ByValueInRange(float(vb), minValue=mn_v, maxValue=mx_v, colorScale="thermal"))
                    cd = Topology.Dictionary(cell)
                    cd = Dictionary.SetValueAtKey(cd, 'vis_color', col)
                    Topology.SetDictionary(cell, cd)
                    vis_cells.append(cell)

            print(f'  Heatmap: {len(vis_cells)} cells')
            fig = Topology.Show(vis_cells,
                          showFigure=False,
                          faceColorKey='vis_color',
                          faceOpacity=1,
                          showEdges=False,
                          showVertices=False,
                          backgroundColor='black',
                          width=900, height=600,
                          renderer=renderer)
            show_ortho(fig)
else:
    print('Isovist analysis skipped (set RUN_ISOVISTS = True to enable).')

## 20. Summary

| Metric | What it reveals |
|--------|----------------|
| **Degree Centrality** | Rooms with the most direct connections — high values at corridor/open-plan areas |
| **Closeness Centrality** | Accessibility — which spaces can reach everywhere most efficiently (global integration) |
| **Betweenness Centrality** | Circulation bottlenecks — spaces that control movement flow between others (choice) |
| **Clustering Coefficient** | Spatial hierarchy — near-zero on grid confirms linear/sequential layout |
| **Shortest Path (red)** | Topological path through graph edges (grid steps) |
| **Shortest Path (blue)** | Geometrically straightened walking path — actual distance savings |
| **Community Detection** | Functional zones identified by Louvain method — reveals spatial groupings |
| **Graph Diameter** | Maximum eccentricity — measures how spread out the spatial network is |
| **Visibility / Isovists** | How much of each floor is visible from each point |

**Three-floor observations:**
- Stair locations should show elevated **betweenness** (they are the only cross-floor connections)
- Stair nodes should show moderate **closeness** (they bridge all three floors)
- **Community detection** may split along floor boundaries or reveal cross-floor zones connected by stairs
- With 3 floors, the **diameter** should be significantly larger than the 2-floor Type K — longer vertical traversal